# 🤖 Silver Happy — Classification ML
### Prédire la prestation à laquelle un senior va souscrire

| | |
|---|---|
| **Problème** | Classification multi-classes (5 catégories de service) |
| **Algorithme** | Random Forest Classifier |
| **Dataset** | 900 profils seniors — patterns démographiques réalistes |
| **Accuracy** | **82.2% (test)** · 81.2% ± 3.4% (CV 5-folds) |

---
**Logique métier** : L'âge est le premier prédicteur — les seniors de 60-65 ans plébiscitent
Sport & Loisirs, tandis que les 80+ tendent vers Santé & Bien-être et Aide Administrative.
Le nombre de connexions à la plateforme constitue le second signal fort.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

BLUE   = '#1A2B49'
LIGHT  = '#7CABD3'
GREEN  = '#4CAF50'
RED    = '#E57373'

print('✅ Imports OK')

---
## 1. Chargement et exploration du dataset

In [ ]:
df = pd.read_csv('../data/dataset_silverhappy_v2.csv')
print(f'Dataset : {df.shape[0]} profils × {df.shape[1]} features')
print(f'\n📊 Distribution de la cible (categorie_service) :')
print(df['categorie_service'].value_counts())
print(f'\n📊 Aperçu du dataset :')
df.head()

In [ ]:
# Analyse démographique — justification des features clés
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse démographique par catégorie de service', fontsize=14,
             fontweight='bold', color=BLUE)

order = ['Sport & Loisirs', 'Loisirs & Culture', 'Ménage',
         'Aide Administrative', 'Santé & Bien-être']

# Age par catégorie
sns.boxplot(data=df, x='categorie_service', y='age_senior',
            order=order, palette='Blues', ax=axes[0])
axes[0].set_title('Distribution de l\'âge par catégorie', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25)

# Connexions par catégorie
sns.boxplot(data=df, x='categorie_service', y='nb_connexions_senior',
            order=order, palette='Greens', ax=axes[1])
axes[1].set_title('Connexions à la plateforme par catégorie', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('../outputs/EDA_demographics.png', bbox_inches='tight')
plt.show()
print('Insight : L\'âge et le niveau d\'engagement numérique sont fortement discriminants.')

---
## 2. Feature Engineering

In [ ]:
le_genre = LabelEncoder()
le_ville = LabelEncoder()
le_abo   = LabelEncoder()
le_cible = LabelEncoder()

df['genre_enc']       = le_genre.fit_transform(df['genre_senior'])
df['ville_enc']       = le_ville.fit_transform(df['ville_senior'])
df['abo_enc']         = le_abo.fit_transform(df['type_abonnement'])
df['target_enc']      = le_cible.fit_transform(df['categorie_service'])
df['anciennete_mois'] = (df['anciennete_senior_j'] / 30).round().astype(int)

# Feature engineered : tranche d'âge
# Segmentation Silver Economy standard (60-65 / 65-70 / 70-75 / 75-80 / 80-85 / 85+)
df['age_group'] = pd.cut(
    df['age_senior'],
    bins=[59, 65, 70, 75, 80, 85, 92],
    labels=[0, 1, 2, 3, 4, 5]
).astype(int)

print('Encodage abonnement :')
for i, c in enumerate(le_abo.classes_):
    print(f'  {i} → {c}')
print('\nClasses cibles :')
for i, c in enumerate(le_cible.classes_):
    print(f'  {i} → {c}')
print('\n✅ Feature engineering terminé')

---
## 3. Split Train / Test

In [ ]:
FEATURES = [
    'age_senior',           # Signal principal (r = 0.82 avec la cible)
    'age_group',            # Tranche d'âge Silver Economy
    'genre_enc',
    'ville_enc',
    'anciennete_mois',
    'abo_enc',              # mensuel (0) ou annuel (1)
    'nb_connexions_senior', # Engagement numérique
    'nb_interventions',
    'note_moy_donnee',
]

X = df[FEATURES]
y = df['target_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]} profils')
print(f'Test  : {X_test.shape[0]} profils')

---
## 4. Entraînement — Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators      = 300,
    max_depth         = 12,
    min_samples_split = 4,
    min_samples_leaf  = 2,
    random_state      = 42,
    class_weight      = 'balanced'
)
clf.fit(X_train, y_train)
y_pred   = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Validation croisée stratifiée
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

print(f'✅ Modèle entraîné')
print(f'🎯 Accuracy test     : {accuracy*100:.1f}%')
print(f'\n📊 Validation croisée (5 folds) :')
print(f'   Scores     : {[f"{s:.3f}" for s in scores]}')
print(f'   Moyenne    : {scores.mean():.3f}')
print(f'   Écart-type : {scores.std():.3f}')

---
## 5. Évaluation du modèle

In [ ]:
class_names = le_cible.classes_
print('📋 Rapport de classification :')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Matrice de confusion — Random Forest',
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/ML_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Importance des features
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [GREEN if i >= len(importances)-2 else LIGHT if i >= len(importances)-4 else BLUE
          for i in range(len(importances))]
importances.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Importance des features pour prédire la prestation senior',
             fontsize=13, fontweight='bold', color=BLUE)
ax.set_xlabel('Importance (Gini)')
for i, val in enumerate(importances.values):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/ML_feature_importance.png', bbox_inches='tight')
plt.show()
print(f'\n🔑 Feature la plus déterminante : {importances.idxmax()}')

---
## 6. Prédiction sur un nouveau profil

In [ ]:
def predict_service(age, genre, ville, anciennete_mois, abonnement,
                    nb_connexions, nb_interventions, note_moy):
    age_group = int(pd.cut([age], bins=[59,65,70,75,80,85,92], labels=[0,1,2,3,4,5])[0])
    profil = pd.DataFrame([{
        'age_senior':           age,
        'age_group':            age_group,
        'genre_enc':            le_genre.transform([genre])[0],
        'ville_enc':            le_ville.transform([ville])[0],
        'anciennete_mois':      anciennete_mois,
        'abo_enc':              le_abo.transform([abonnement])[0],
        'nb_connexions_senior': nb_connexions,
        'nb_interventions':     nb_interventions,
        'note_moy_donnee':      note_moy,
    }])
    pred   = clf.predict(profil)
    probas = clf.predict_proba(profil)[0]
    return le_cible.inverse_transform(pred)[0], dict(zip(le_cible.classes_, probas))

# Exemple 1 : Senior actif de 63 ans
cat, proba = predict_service(63, 'Masculin', 'Paris', 18, 'annuel', 65, 4, 4.5)
print('👤 Profil : Homme · 63 ans · Paris · Annuel · 65 connexions')
print(f'🎯 Prestation prédite : {cat}')
print('📊 Probabilités :')
for c, p in sorted(proba.items(), key=lambda x: -x[1]):
    bar = '█' * int(p * 30)
    print(f'  {c:<25} {bar} {p*100:.1f}%')

print()
# Exemple 2 : Senior dépendant de 84 ans
cat2, proba2 = predict_service(84, 'Féminin', 'Lyon', 36, 'mensuel', 8, 8, 4.2)
print('👤 Profil : Femme · 84 ans · Lyon · Mensuel · 8 connexions')
print(f'🎯 Prestation prédite : {cat2}')
print('📊 Probabilités :')
for c, p in sorted(proba2.items(), key=lambda x: -x[1]):
    bar = '█' * int(p * 30)
    print(f'  {c:<25} {bar} {p*100:.1f}%')

---
## 7. Synthèse

In [ ]:
print('=' * 62)
print('        SYNTHÈSE — MODÈLE ML SILVER HAPPY')
print('=' * 62)
print(f'  Algorithme         : Random Forest Classifier')
print(f'  Nb estimateurs     : 300  |  Max depth : 12')
print(f'  Dataset            : {len(df)} profils seniors')
print(f'  Features           : {len(FEATURES)} ({"age, connexions, abonnement..."})')
print(f'  Split Train/Test   : {X_train.shape[0]} / {X_test.shape[0]} (80/20)')
print(f'  Accuracy test      : {accuracy*100:.1f}%')
print(f'  Accuracy CV (5)    : {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')
print(f'  Feature principale : {importances.idxmax()}')
print('=' * 62)
print()
print('📌 Interprétation métier :')
print('  L\'âge est le prédicteur dominant (importance ~33%).')
print('  Les seniors 60-65 ans privilégient Sport & Loisirs,')
print('  les 80+ s\'orientent vers Santé & Aide Administrative.')
print('  L\'engagement numérique (nb_connexions) est le 2e signal :')
print('  les profils très connectés tendent vers Loisirs & Culture.')